<a href="https://colab.research.google.com/github/mc-castro/clinicaltrials-ia-thesis/blob/mc-castro%2Fdevelop/notebooks/transform_protocols.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
import pandas as pd
import ast
from google.colab import userdata, drive
from tqdm import tqdm
import typing_extensions as typing
import google.generativeai as genai
import json
import time
import os
import csv


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Extract Data

In [ ]:
# 1. Load the data and convert strings back to real Python lists
clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")
# clinical_trials_df['inclusion_criteria'] = clinical_trials_df['inclusion_criteria'].apply(ast.literal_eval)

In [ ]:
mimic_patients_df = pd.read_parquet("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet")
print(mimic_patients_df.columns.tolist())

['subject_id', 'hadm_id', 'HPI', 'physical_exam', 'chief_complaint', 'icd_diag_principal', 'name_diag_principal', 'list_procedures', '% Hemoglobin A1c', '25-OH Vitamin D', 'ADP', 'ANCA Titer', 'ARCH-1', 'Absolute Basophil Count', 'Absolute CD3 Count', 'Absolute CD4 Count', 'Absolute CD8 Count', 'Absolute Eosinophil Count', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 'Absolute Neutrophil Count', 'Acanthocytes', 'Acetaminophen', 'Acetone', 'Alanine Aminotransferase (ALT)', 'Albumin', 'Alkaline Phosphatase', 'Alpha-Fetoprotein', 'Alveolar-arterial Gradient', 'Ammonia', 'Amylase', 'Anion Gap', 'Anisocytosis', 'Anti-Mitochondrial Antibody', 'Anti-Neutrophil Cytoplasmic Antibody', 'Anti-Nuclear Antibody', 'Anti-Nuclear Antibody, Titer', 'Anti-Smooth Muscle Antibody', 'Anti-Thyroglobulin Antibodies', 'Anticardiolipin Antibody IgG', 'Anticardiolipin Antibody IgM', 'Antithrombin', 'Arachadonic Acid', 'Asparate Aminotransferase (AST)', 'Assist/Control', 'Atypical Lymphocytes', 'Bands

In [ ]:
len(mimic_patients_df)

2813

## Transform

In [ ]:
# @title Clinical Eligibility Engine
class ClinicalEligibilityEngine:
    def __init__(self, api_key, version_genai, mimic_columns):
        """
        Initializes the engine with the API and the available MIMIC columns.
        """
        genai.configure(api_key=userdata.get(api_key))
        self.model = genai.GenerativeModel(version_genai)
        self.mimic_columns = mimic_columns

    # --- FASE 1: TRADUTOR IA (Com instrução de Diagnóstico) ---
    def _call_ai_translator(self, inclusion, exclusion):
        prompt = f"""
        Analyze clinical trial criteria and map them to these columns: {self.mimic_columns}

        SPECIFIC INSTRUCTIONS:
        1. AGE & GENDER: Use 'age' (numeric) and 'gender' (M/F).
        2. CLINICAL CONDITIONS: For any disease, symptom, or diagnosis (e.g., 'Diabetes', 'Pleural Effusion'),
           map it to 'HPI' as the target_column, but set type to 'text'.
           The system will automatically search in 'name_diag_principal' and 'chief_complaint' as well.
        3. OPERATORS: Use '>', '<', '>=', '<=', '==', 'contains', or 'not_contains'.
        4. GENDER VALUES: Always use 'M' or 'F' for gender values.
        5. IGNORE: Administrative criteria like 'Informed consent' or 'Hospital admission'.

        INPUT:
        Inclusion: {inclusion}
        Exclusion: {exclusion}

        RETURN ONLY JSON:
        {{
          "inclusion_rules": [{{"column": "...", "op": "...", "value": "...", "type": "..."}}],
          "exclusion_rules": [...]
        }}
        """
        try:
            response = self.model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
            return response.text
        except:
            return None

    # --- FASE 1: GERAÇÃO DO LIVRO DE REGRAS (AI) ---
    def build_rulebook(self, studies_df, output_file="study_rulebook.csv"):
        """
        Transforma critérios de texto em regras lógicas estruturadas num CSV.
        """
        if os.path.exists(output_file):
            rulebook_df = pd.read_csv(output_file)
            print(f"A retomar: {len(rulebook_df)} estudos já processados.")
        else:
            rulebook_df = pd.DataFrame(columns=["nct_id", "rules_json"])

        processed_ncts = rulebook_df['nct_id'].tolist()

        for _, row in tqdm(studies_df.iterrows(), total=len(studies_df), desc="Fase 1: Tradução IA"):
            nct_id = row['nct_id']
            if nct_id in processed_ncts:
                continue

            rules_json = self._call_ai_translator(row['inclusion_criteria'], row['exclusion_criteria'])

            if rules_json:
                new_entry = pd.DataFrame([{"nct_id": nct_id, "rules_json": rules_json}])
                rulebook_df = pd.concat([rulebook_df, new_entry], ignore_index=True)
                rulebook_df.to_csv(output_file, index=False)

            time.sleep(4) # Delay para evitar bloqueio da API gratuita


    # --- FASE 2: MATCHING LOCAL (Com busca em múltiplas colunas) ---
    def _get_mask(self, df, rule, invert=False):
        col = rule.get('column')
        op = rule.get('op')
        val = rule.get('value')
        type_ = str(rule.get('type')).lower() # Normaliza para evitar erro number/numeric

        # Se a coluna não existe no DF e não é texto, passamos o filtro (True)
        if col not in df.columns and type_ not in ['text', 'string']:
            return pd.Series([True] * len(df))

        m = pd.Series([True] * len(df))
        try:
            # 1. TRATAMENTO DE GÊNERO (M/F)
            if col.lower() == 'gender':
                first_letter = str(val)[0].upper()
                m = df['gender'].str.upper().str.startswith(first_letter, na=False)

            # 2. TRATAMENTO NUMÉRICO (age, labs, etc.)
            elif type_ in ['numeric', 'number']:
                v = pd.to_numeric(val, errors='coerce')
                series = pd.to_numeric(df[col], errors='coerce')
                if op == '>': m = (series > v)
                elif op == '<': m = (series < v)
                elif op == '>=': m = (series >= v)
                elif op == '<=': m = (series <= v)
                elif op == '==': m = (series == v)

            # 3. TRATAMENTO DE TEXTO (HPI + Diagnóstico + Queixa + Exame Físico)
            elif type_ in ['text', 'string']:
                search_val = str(val).lower()
                # Lista de colunas onde vamos procurar a condição clínica
                text_cols = ['HPI', 'name_diag_principal', 'chief_complaint', 'physical_exam']

                # Inicializa máscara como False e vai somando (OR)
                m = pd.Series([False] * len(df))
                for t_col in text_cols:
                    if t_col in df.columns:
                        m |= df[t_col].astype(str).str.lower().str.contains(search_val, na=False)

                if op == 'not_contains':
                    m = ~m
        except Exception as e:
            # Em caso de erro, assume que o paciente passa (não bloqueia por erro de dado)
            return pd.Series([True] * len(df))

        return ~m if invert else m

    def run_matching(self, mimic_df, rulebook_file, final_output="final_matching_report.csv"):
        # Carregamento robusto: ignora linhas malformadas no CSV para não travar o script todo
        try:
            rulebook = pd.read_csv(rulebook_file, on_bad_lines='skip')
        except Exception as e:
            print(f"Erro crítico ao ler o arquivo CSV: {e}")
            return None

        results = []

        print(f"Fase 2: Cruzamento de Dados. Processando {len(rulebook)} regras válidas...")
        for _, study in tqdm(rulebook.iterrows(), total=len(rulebook), desc="A filtrar pacientes"):
            nct_id = study['nct_id']
            raw_json = str(study['rules_json']).strip()

            try:
                # Tenta limpar problemas comuns de escape antes de decodificar
                # Remove possíveis aspas extras no início/fim e normaliza
                if raw_json.startswith("'") and raw_json.endswith("'"):
                    raw_json = raw_json[1:-1]

                rules = json.loads(raw_json)
            except Exception as e:
                # Se ainda assim falhar, tentamos uma correção agressiva para aspas simples
                try:
                    rules = ast.literal_eval(raw_json)
                except:
                    # Se falhar totalmente, pula este estudo
                    results.append({"nct_id": nct_id, "eligible_count": 0, "subject_ids": "[]"})
                    continue

            mask = pd.Series([True] * len(mimic_df))

            for r in rules.get('inclusion_rules', []):
                mask &= self._get_mask(mimic_df, r, invert=False)

            for r in rules.get('exclusion_rules', []):
                mask &= self._get_mask(mimic_df, r, invert=True)

            eligible_ids = mimic_df.loc[mask, 'subject_id'].tolist()
            results.append({
                "nct_id": nct_id,
                "eligible_count": len(eligible_ids),
                "subject_ids": json.dumps(eligible_ids)
            })

        final_df = pd.DataFrame(results)
        final_df.to_csv(final_output, index=False)
        return final_df

In [ ]:
## Main

In [ ]:
# @title Define Engine
all_cols = mimic_patients_df.columns.tolist()
to_ignore = ['subject_id', 'hadm_id']
filtered_cols = [c for c in all_cols if c not in to_ignore and not c.endswith('_FLAG')]

engine = ClinicalEligibilityEngine(
    api_key="ParsingGeminiAPI",
    version_genai = 'gemini-2.5-flash',
    mimic_columns=filtered_cols
)

In [ ]:
# @title Create rulebook
engine.build_rulebook(clinical_trials_df, output_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/rulebook.csv")

In [ ]:
input_file = "/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/rulebook.csv"
output_file = "/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/rulebook_cleaned.csv"

with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8', newline='') as f_out:
    reader = csv.reader(f_in)
    writer = csv.writer(f_out)

    header = next(reader)
    writer.writerow(header)

    for i, row in enumerate(reader):
        try:
            # Tenta validar se o JSON na segunda coluna é minimamente carregável
            json.loads(row[1])
            writer.writerow(row)
        except:
            print(f"Linha {i} descartada por JSON corrompido no CSV.")

Linha 359 descartada por JSON corrompido no CSV.
Linha 368 descartada por JSON corrompido no CSV.
Linha 474 descartada por JSON corrompido no CSV.


In [ ]:
# @title Matching Patients
report = engine.run_matching(mimic_patients_df,
                             rulebook_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/rulebook_cleaned.csv",
                             final_output="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/final_matching_report.csv")

print(f"Sucesso! Encontrados pacientes para {len(report[report['eligible_count'] > 0])} estudos.")

Fase 2: Cruzamento de Dados (Processamento Local)...


A filtrar pacientes:   0%|          | 2/997 [00:00<02:16,  7.31it/s]/tmp/ipython-input-3438429809.py:107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m |= df[t_col].astype(str).str.lower().str.contains(search_val, na=False)
/tmp/ipython-input-3438429809.py:107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m |= df[t_col].astype(str).str.lower().str.contains(search_val, na=False)
/tmp/ipython-input-3438429809.py:107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m |= df[t_col].astype(str).str.lower().str.contains(search_val, na=False)
/tmp/ipython-input-3438429809.py:107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m |= df[t_col].ast

Sucesso! Encontrados pacientes para 136 estudos.
